<a href="https://colab.research.google.com/github/Sairika/financial-policy-chatbot/blob/main/financial_policy_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Document Question Answering with RAG

This notebook demonstrates how to build a simple Retrieval-Augmented Generation (RAG) system using Langchain to answer questions based on a PDF document.

## Setup

The initial cells install necessary libraries and import the required modules.

In [ ]:
# Install necessary libraries
%%capture
!pip install --user "langchain==0.1.16"
!pip install --user "langchain-ibm==0.1.4"
!pip install --user "huggingface == 0.0.1"
!pip install --user "huggingface-hub == 0.23.4"
!pip install --user "sentence-transformers == 2.5.1"
!pip install --user "chromadb"
!pip install --user "wget == 3.2"
!pip install --user "langchain-community"
!pip install --user "pypdf"

### Suppress warnings

In [ ]:
# Suppress warnings
# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
  pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

### Import core Langchain components

In [ ]:
# Import core Langchain components
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

## Document Loading and Processing

This section handles loading your document, splitting it into manageable chunks, creating numerical representations (embeddings), and storing them for efficient retrieval.

### Load and split the PDF document

**Make sure to update the `pdf_filename` to your PDF file path.**

In [ ]:
# Load and split the PDF document
from langchain.document_loaders import PyPDFLoader

# Assuming your PDF file is named finance.pdf and is in the /content/ directory
pdf_filename = '/content/Policy-file.pdf'

loader = PyPDFLoader(pdf_filename)
documents = loader.load()

# You can still use the CharacterTextSplitter or choose a different splitter if needed
text_splitter = CharacterTextSplitter(chunk_size=1500, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

print(len(texts))

### Create embeddings and build a vector store

This step converts the text chunks into numerical vectors and stores them in a vector database for quick searching.

In [ ]:
# Create embeddings and build a vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
collection_name_final = "policy_document_embeddings_final"
docsearch = Chroma.from_documents(
    texts,
    embeddings,
    collection_name=collection_name_final
)
print('Document ingested with "sentence-transformers/all-MiniLM-L6-v2" embeddings')

## Language Model Setup

This section loads the language model that will be used to generate answers based on the retrieved document chunks.

### Load the Language Model (LLM) and set up the pipeline

This loads the specified Hugging Face model and prepares it for text generation.

In [ ]:
# Load the Language Model (LLM) and set up the pipeline
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# Choose the Llama-3.1-8B-Instruct model
hf_model_id = "meta-llama/Llama-3.1-8B-Instruct"

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(hf_model_id)
model = AutoModelForCausalLM.from_pretrained(
    hf_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    # load_in_4bit=True # Add this line to load the model in 4-bit quantization
)

# Create a text generation pipeline
pipe = pipeline(
    "text-generation", # Use "text-generation" for Llama models
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512, # Increased tokens for potentially longer responses
    temperature=0.7, # Adjusted temperature for slightly more varied output
    # Add other parameters as needed
)

# Create the HuggingFacePipeline Langchain object
llm = HuggingFacePipeline(pipeline=pipe)

## RAG Chain Configuration

This section sets up the Langchain components that combine the retriever and the LLM.

### Define the prompt template

This defines the instructions given to the language model when generating an answer based on the retrieved context.

In [ ]:
prompt_template = """Use the following pieces of context to answer the question at the end. Strive to provide a comprehensive and insightful answer based *only* on the provided text.

Context: {context}

Question: {question}

Concise Answer:"""
PROMPT = PromptTemplate(
template=prompt_template, input_variables=["context", "question"]
)
# Removed chain_type_kwargs as we are not using it in the qa() function anymore
# chain_type_kwargs = {"prompt": PROMPT}

### Configure RetrievalQA chain (for single-turn questions)

This sets up a chain for simple question answering. Note that the interactive agent uses a different chain (`ConversationalRetrievalChain`).

In [ ]:
# RetrievalQA Chain (for single-turn questions)
qa_retrieval = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    chain_type_kwargs=chain_type_kwargs, # Use the enhanced prompt
    return_source_documents=False
)
print("Configured RetrievalQA chain.")

### Configure ConversationalRetrieval Chain (for multi-turn questions)

This sets up the chain used by the interactive agent, allowing for follow-up questions by maintaining conversation history.

In [ ]:
# ConversationalRetrieval Chain (for multi-turn questions)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
qa_conversational = ConversationalRetrievalChain.from_llm(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    memory=memory,
    get_chat_history=lambda h: h,
    return_source_documents=False,
)
print("Configured ConversationalRetrieval chain.")

### Define queries for testing the RetrievalQA system

In [ ]:
# Define queries for testing the RetrievalQA system
print("\n--- Testing Enhanced RAG System (RetrievalQA) ---")
queries_to_test = [
    "what is debt policy?",
    "Tell me the total budget?",
    "What is the purpose of the financial policy objectives and strategies statement?",
]

### Test the RetrievalQA chain with defined queries

In [ ]:
# Test the RetrievalQA chain with defined queries
retrieval_results = {}
for query in queries_to_test:
    print(f"\nQuery: {query}")
    try:
        result = qa_retrieval.invoke({"query": query})
        answer = result["result"]
        retrieval_results[query] = answer
        print(f"Answer: {answer}")
    except Exception as e:
        print(f"An error occurred: {e}")
        retrieval_results[query] = f"Error: {e}"

print("\n--- Testing Enhanced RAG System (ConversationalRetrievalChain) ---")

### Reset memory and define queries for testing the ConversationalRetrievalChain

In [ ]:
# Reset memory for conversational testing
memory.clear()
conversational_results = {}
chat_history = []

conversational_queries = [
    "What are the strategic priorities related to the Budget?",
    "Tell me more about the debt policy.",
]

### Test the ConversationalRetrievalChain with defined queries

In [ ]:
# Test the ConversationalRetrievalChain with defined queries
for query in conversational_queries:
    print(f"\nQuery: {query}")
    try:
        # For ConversationalRetrievalChain, pass question and chat_history
        result = qa_conversational.invoke({"question": query, "chat_history": chat_history})
        answer = result["answer"]
        conversational_results[query] = answer
        print(f"Answer: {answer}")
        # Update chat history
        chat_history.append((query, answer))
    except Exception as e:
        print(f"An error occurred: {e}")
        conversational_results[query] = f"Error: {e}"
        # Still append to history to see if subsequent turns are affected
        chat_history.append((query, f"Error: {e}"))

# Note: Manual analysis of the outputs is needed to compare performance.

## Interactive Agent

This section defines and runs the interactive agent function that allows for conversational questioning of the document.

### Define the qa() function for the interactive conversational agent

This function sets up and runs the interactive session using the `ConversationalRetrievalChain` and the defined prompt.

In [ ]:
def qa():
    memory = ConversationBufferMemory(memory_key = "chat_history", return_message = True)
    # Ensure PROMPT is defined from the latest prompt template modification in cell 191nZvIL-E53
    # Ensure llm and docsearch are defined from previous cells
    qa = ConversationalRetrievalChain.from_llm(llm=llm,
                                                chain_type="stuff",
                                                retriever=docsearch.as_retriever(),
                                                memory = memory,
                                                get_chat_history=lambda h : h,
                                                return_source_documents=False,
                                                combine_docs_chain_kwargs={"prompt": PROMPT}) # Using combine_docs_chain_kwargs with simplified PROMPT

    history = []
    print("Ask a question about the document (type 'quit', 'exit', or 'bye' to end):")
    while True:
        query = input("Question: ")
        if query.lower() in ["quit","exit","bye"]:
            print("Answer: Goodbye!")
            break
        # Check if docsearch is defined before invoking
        if 'docsearch' not in locals() and 'docsearch' not in globals():
            print("Error: docsearch is not defined. Please run the cell to load and process the document first.")
            continue
        # Check if llm is defined before invoking
        if 'llm' not in locals() and 'llm' not in globals():
            print("Error: llm is not defined. Please run the cell to load the language model first.")
            continue
        # Check if PROMPT is defined before invoking
        if 'PROMPT' not in locals() and 'PROMPT' not in globals():
             print("Error: PROMPT is not defined. Please run the cell to define the prompt template first.")
             continue


        result = qa.invoke({"question": query, "chat_history": history})
        raw_answer = result["answer"]

        # Removed post-processing logic for simplicity as requested.
        # The raw output from the model will now be printed directly.
        concise_answer = raw_answer


        history.append((query, raw_answer)) # Append the raw answer to history for potentially better context in future turns
        print("Answer: ", concise_answer)

### Execute the qa() function to start the interactive conversational agent

Run this cell to begin the interactive question-answering session.

In [ ]:
qa()